# Train Model

## 1. SetUp

In [1]:
import os
import sys
# sys.path.append('/Users/zhenyvlu/work/sesame')
sys.path.append(r"C:\Users\luzhe\deep_learning\sesame")

import numpy as np
import torch

%matplotlib inline
import matplotlib.pyplot as plt

import sesame.utils.distributed as du
import sesame.utils.checkpoint as cu
import sesame.utils.misc as misc
import sesame.models.optimizer as optim
import sesame.visualization.tensorboard_vis as tb
import sesame.utils.logger as logger

from sesame.utils.parser import load_config, parse_args
from sesame.datasets.kinetics import Kinetics
from sesame.models.model_builder import MViT
from sesame.utils.meters import TrainMeter, ValMeter, EpochTimer
from sesame.config.defaults import get_cfg
from tools.train_net import train_epoch, eval_epoch

log = logger.get_logger('notebook')
logger.setup_logging('.')

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

## 2. Build model

In [2]:
cfg = get_cfg()
cfg.merge_from_file("../configs/Kinetics/MINI_MVIT_B_16x4_CONV.yaml")

# cfg = CN.load_cfg(open("../configs/Kinetics/MVIT_B_16x4_CONV.yaml", "r", encoding="UTF-8"))

In [9]:
model = MViT(cfg)
checkpoint = torch.load(r"C:\Users\luzhe\deep_learning\multiscale vision transformers\K400_MVIT_B_16x4_CONV.pyth", map_location=torch.device('cpu'))
# 去除头部权重
model_dict = checkpoint['model_state']
# model_dict.keys()
model_dict.pop('head.projection.weight')
model_dict.pop('head.projection.bias')

model.load_state_dict(model_dict, strict=False)

_IncompatibleKeys(missing_keys=['blocks.14.attn.pool_k.weight', 'blocks.14.attn.norm_k.weight', 'blocks.14.attn.norm_k.bias', 'blocks.14.attn.pool_v.weight', 'blocks.14.attn.norm_v.weight', 'blocks.14.attn.norm_v.bias', 'blocks.15.attn.pool_k.weight', 'blocks.15.attn.norm_k.weight', 'blocks.15.attn.norm_k.bias', 'blocks.15.attn.pool_v.weight', 'blocks.15.attn.norm_v.weight', 'blocks.15.attn.norm_v.bias', 'head.projection.weight', 'head.projection.bias'], unexpected_keys=[])

In [4]:
for param in model.parameters():
    param.requires_grad = False
    
# 修改梯度下降
model.blocks[14].attn.pool_k.weight.requires_grad = True
model.blocks[14].attn.norm_k.weight.requires_grad = True
model.blocks[14].attn.norm_k.bias.requires_grad = True

model.blocks[14].attn.pool_v.weight.requires_grad = True
model.blocks[14].attn.norm_v.weight.requires_grad = True
model.blocks[14].attn.norm_v.bias.requires_grad = True

model.blocks[15].attn.pool_k.weight.requires_grad = True
model.blocks[15].attn.norm_k.weight.requires_grad = True
model.blocks[15].attn.norm_k.bias.requires_grad = True

model.blocks[15].attn.pool_v.weight.requires_grad = True
model.blocks[15].attn.norm_v.weight.requires_grad = True
model.blocks[15].attn.norm_v.bias.requires_grad = True

model.head.projection.weight.requires_grad = True
model.head.projection.bias.requires_grad = True

# missing_keys=['blocks.14.attn.pool_k.weight', 
#               'blocks.14.attn.norm_k.weight', 
#               'blocks.14.attn.norm_k.bias', 
              
#               'blocks.14.attn.pool_v.weight', 
#               'blocks.14.attn.norm_v.weight', 
#               'blocks.14.attn.norm_v.bias', 
              
#               'blocks.15.attn.pool_k.weight', 
#               'blocks.15.attn.norm_k.weight', 
#               'blocks.15.attn.norm_k.bias', 
              
#               'blocks.15.attn.pool_v.weight', 
#               'blocks.15.attn.norm_v.weight', 
#               'blocks.15.attn.norm_v.bias']

In [5]:
model.to(device)

MViT(
  (patch_embed): PatchEmbed(
    (proj): Conv3d(3, 96, kernel_size=(3, 7, 7), stride=(2, 4, 4), padding=(1, 3, 3))
  )
  (blocks): ModuleList(
    (0): MultiScaleBlock(
      (norm1): LayerNorm((96,), eps=1e-06, elementwise_affine=True)
      (attn): MultiScaleAttention(
        (qkv): Linear(in_features=96, out_features=288, bias=True)
        (proj): Linear(in_features=96, out_features=96, bias=True)
        (pool_k): Conv3d(96, 96, kernel_size=(3, 3, 3), stride=(1, 8, 8), padding=(1, 1, 1), groups=96, bias=False)
        (norm_k): LayerNorm((96,), eps=1e-05, elementwise_affine=True)
        (pool_v): Conv3d(96, 96, kernel_size=(3, 3, 3), stride=(1, 8, 8), padding=(1, 1, 1), groups=96, bias=False)
        (norm_v): LayerNorm((96,), eps=1e-05, elementwise_affine=True)
      )
      (drop_path): Identity()
      (norm2): LayerNorm((96,), eps=1e-06, elementwise_affine=True)
      (mlp): Mlp(
        (fc1): Linear(in_features=96, out_features=384, bias=True)
        (act): GELU()
 

## 3. Optimizer

In [6]:
optimizer = optim.construct_optimizer(model, cfg)

[01/07 11:10:55][INFO] optimizer.py:  74: bn 0, non bn 4, zero 8, no grad 301


## 4. Load Data

In [7]:
kinetics_dataset_train = Kinetics(cfg, 'train')
kinetics_dataset_val = Kinetics(cfg, 'val')

[01/07 11:10:57][INFO] kinetics.py: 113: Constructing Kinetics train...
[01/07 11:10:59][INFO] kinetics.py: 159: Constructing kinetics dataloader (size: 234619) from E:\datasets\k400\index\train.txt
[01/07 11:10:59][INFO] kinetics.py: 113: Constructing Kinetics val...
[01/07 11:11:00][INFO] kinetics.py: 159: Constructing kinetics dataloader (size: 19761) from E:\datasets\k400\index\val.txt


In [8]:
kinetics_dataset_train_loader = torch.utils.data.DataLoader(
            kinetics_dataset_train,
            batch_size=int(cfg.TRAIN.BATCH_SIZE / max(1, cfg.NUM_GPUS)),
            num_workers=cfg.DATA_LOADER.NUM_WORKERS,
            pin_memory=cfg.DATA_LOADER.PIN_MEMORY,
            shuffle=True,
            drop_last=True,
            worker_init_fn=None,
        )

kinetics_dataset_val_loader = torch.utils.data.DataLoader(
            kinetics_dataset_val,
            batch_size=int(cfg.TRAIN.BATCH_SIZE / max(1, cfg.NUM_GPUS)),
            num_workers=cfg.DATA_LOADER.NUM_WORKERS,
            pin_memory=cfg.DATA_LOADER.PIN_MEMORY,
            drop_last=True,
            worker_init_fn=None,
        )

## 5. Build Meters

In [9]:
train_meter = TrainMeter(len(kinetics_dataset_train_loader), cfg)
val_meter = ValMeter(len(kinetics_dataset_val_loader), cfg)

## 4. Training

In [ ]:
scaler = torch.cuda.amp.GradScaler(enabled=cfg.TRAIN.MIXED_PRECISION)

# set up writer for logging to Tensorboard format.
if cfg.TENSORBOARD.ENABLE and du.is_master_proc(cfg.NUM_GPUS * cfg.NUM_SHARDS):
    writer = tb.TensorboardWriter(cfg)
else:
    writer = None

start_epoch = -1
# Perform the training loop.
# log.info("Start epoch: {}".format(start_epoch + 1))

epoch_timer = EpochTimer()
for cur_epoch in range(start_epoch, cfg.SOLVER.MAX_EPOCH):
    # Shuffle the dataset.
#     loader.shuffle_dataset(train_loader, cur_epoch)

    # Train for one epoch.
    epoch_timer.epoch_tic()
    train_epoch(
        kinetics_dataset_train_loader,
        model,
        optimizer,
        scaler,
        train_meter,
        cur_epoch,
        cfg,
        writer,
    )
    epoch_timer.epoch_toc()

    is_checkp_epoch = cu.is_checkpoint_epoch(
        cfg,
        cur_epoch,
        None
    )
    is_eval_epoch = misc.is_eval_epoch(
        cfg, cur_epoch, None
    )

    # Compute precise BN stats.
    if (
        (is_checkp_epoch or is_eval_epoch)
        and cfg.BN.USE_PRECISE_STATS
        and len(get_bn_modules(model)) > 0
    ):
        calculate_and_update_precise_bn(
            precise_bn_loader,
            model,
            min(cfg.BN.NUM_BATCHES_PRECISE, len(precise_bn_loader)),
            cfg.NUM_GPUS > 0,
        )
#     _ = misc.aggregate_sub_bn_stats(model)

    # Save a checkpoint.
    if is_checkp_epoch:
        cu.save_checkpoint(
            cfg.OUTPUT_DIR,
            model,
            optimizer,
            cur_epoch,
            cfg,
            scaler if cfg.TRAIN.MIXED_PRECISION else None,
        )
    # Evaluate the model on validation set.
    if is_eval_epoch:
        eval_epoch(kinetics_dataset_val_loader, model, val_meter, cur_epoch, cfg, writer)

if writer is not None:
    writer.close()

[11/17 10:47:54][DEBUG] tpu_cluster_resolver.py:  35: Falling back to TensorFlow client; we recommended you install the Cloud TPU client directly with pip install cloud-tpu-client.
[11/17 10:47:55][INFO] tensorboard_vis.py: 249: To see logged results in Tensorboard, please launch using the command                         `tensorboard  --port=<port-number> --logdir E:\demo\runs-kinetics`
[11/17 10:48:41][INFO] logger.py:  81: json_stats: {"_type": "train_iter", "dt": 4.79198, "dt_data": 4.22879, "dt_net": 0.56319, "epoch": "0/200", "eta": "326 days, 22:29:39", "gpu_mem": "2.17G", "iter": "10/29327", "loss": 4.72380, "lr": -0.00000, "top1_err": 37.50000, "top5_err": 12.50000}
[11/17 10:49:22][INFO] logger.py:  81: json_stats: {"_type": "train_iter", "dt": 4.13561, "dt_data": 3.57448, "dt_net": 0.56113, "epoch": "0/200", "eta": "282 days, 3:43:10", "gpu_mem": "2.17G", "iter": "20/29327", "loss": 4.31769, "lr": -0.00000, "top1_err": 37.50000, "top5_err": 25.00000}
[11/17 10:50:02][INFO] lo

[11/17 11:07:17][INFO] logger.py:  81: json_stats: {"_type": "train_iter", "dt": 3.65994, "dt_data": 3.10908, "dt_net": 0.55086, "epoch": "0/200", "eta": "249 days, 16:35:31", "gpu_mem": "2.17G", "iter": "280/29327", "loss": 4.38584, "lr": -0.00000, "top1_err": 37.50000, "top5_err": 25.00000}
[11/17 11:07:58][INFO] logger.py:  81: json_stats: {"_type": "train_iter", "dt": 4.27280, "dt_data": 3.72229, "dt_net": 0.55051, "epoch": "0/200", "eta": "291 days, 12:02:48", "gpu_mem": "2.17G", "iter": "290/29327", "loss": 4.43740, "lr": -0.00000, "top1_err": 43.75000, "top5_err": 25.00000}
[11/17 11:08:37][INFO] logger.py:  81: json_stats: {"_type": "train_iter", "dt": 3.89323, "dt_data": 3.32961, "dt_net": 0.56362, "epoch": "0/200", "eta": "265 days, 14:32:31", "gpu_mem": "2.17G", "iter": "300/29327", "loss": 4.43866, "lr": -0.00000, "top1_err": 50.00000, "top5_err": 25.00000}
[11/17 11:09:19][INFO] logger.py:  81: json_stats: {"_type": "train_iter", "dt": 4.14847, "dt_data": 3.53799, "dt_net"